# 07 · Análise de erro — FN-rate por categoria (Tabela 12)

- **Origem:** **notebook NOVO** (extensão) — o autor não liberou um notebook para a Tabela 12; é o único da pasta que não parte de um original. Autossuficiente.
- **Faz:** reproduz a Tabela 12 — taxa de falsos negativos por categoria fina — cruzando as predições do **M-BERT-BR** no *test* com as categorias do `ToLD-BR.csv`, via uma **chave de texto normalizada** (minúsculas + remoção de acentos + remoção de U+FFFD + colapso de espaços) que recupera 2100/2100 alinhamentos (ver `REPORT.md` §7.2).
- **Entradas:** `results/preds_binary_mbert_br.csv` (gerado pelo `03_classification.ipynb`) e `ToLD-BR.csv`.
- **Saídas:** `results/error_analysis_fn_rate.json` e `results/figures/tab12_fn_rate.png`.

> **Pré-requisito:** rode antes o `03_classification.ipynb` (gera `preds_binary_mbert_br.csv`).

In [2]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "ToLD-BR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_ZIP  = ROOT / "experiments" / "data" / "1annotator.zip"
MAIN_CSV  = ROOT / "ToLD-BR.csv"
ALPHA_CSV = ROOT / "ToLD-BR_alpha.csv"
RESULTS   = ROOT / "reproduction" / "results"
FIGURES   = RESULTS / "figures"
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

In [3]:
import re, json, unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

LABELS = ["homophobia", "obscene", "insult", "racism", "misogyny", "xenophobia"]

## Lógica de alinhamento

`_norm` é a chave de match robusta; `fn_rate_per_label` calcula, por categoria, quantos exemplos
positivos foram preditos como **não-tóxico** (falsos negativos) e a taxa correspondente.

In [4]:
def _norm(s) -> str:
    """Chave de match robusta: os splits binarios diferem do ToLD-BR.csv em caixa, acentos,
    espacos/quebras de linha internas e ~10 linhas com encoding corrompido (U+FFFD)."""
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.replace("\ufffd", "")
    return re.sub(r"\s+", " ", s).strip()


def fn_rate_per_label(preds, cats, labels):
    preds = preds.copy(); cats = cats.copy()
    preds["_key"] = preds["text"].map(_norm)
    cats["_key"] = cats["text"].map(_norm)
    merged = preds.merge(cats.drop_duplicates("_key"), on="_key", how="left")
    out = {}
    for label in labels:
        pos_mask = merged[label] == 1
        positives = int(pos_mask.sum())
        false_neg = int((pos_mask & (merged["y_pred"] == 0)).sum())
        rate = false_neg / positives if positives else 0.0
        out[label] = {"false_negatives": false_neg, "positives": positives, "rate": rate}
    return out

In [5]:
# predicoes do M-BERT-BR (geradas pelo 03_classification) + categorias finas (ToLD-BR.csv binarizado)
preds = pd.read_csv(RESULTS / "preds_binary_mbert_br.csv")
cats = pd.read_csv(MAIN_CSV)
cats[LABELS] = (cats[LABELS] > 0).astype(int)
cats = cats[["text"] + LABELS].dropna().reset_index(drop=True)
print("preds:", len(preds), "| cats:", len(cats))

preds: 2100 | cats: 21000


In [6]:
res = fn_rate_per_label(preds, cats, LABELS)
with open(RESULTS / "error_analysis_fn_rate.json", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print("FN-rate por categoria (Tabela 12):")
for label, v in res.items():
    print(f"  {label:12} {v['false_negatives']:>3}/{v['positives']:<3} ({v['rate']:.2f})")

FN-rate por categoria (Tabela 12):
  homophobia     7/35  (0.20)
  obscene      119/699 (0.17)
  insult        74/444 (0.17)
  racism         7/17  (0.41)
  misogyny       7/44  (0.16)
  xenophobia     9/19  (0.47)


## Comparação com o artigo (Tabela 12) e figura

Os valores do artigo (Leite et al., 2020, Tab. 12) entram como referência. O padrão central
esperado: **classes minoritárias** (racism, xenophobia) têm os **maiores** FN-rates; **majoritárias**
(obscene, insult) os **menores**.

In [7]:
# Tabela 12 do artigo: categoria -> (false_neg, positives, rate)
ARTIGO = {
    "homophobia": (7, 35, 0.20),   # LGBTQ+phobia
    "insult":     (67, 448, 0.15),
    "xenophobia": (13, 19, 0.68),
    "misogyny":   (7, 45, 0.15),
    "obscene":    (117, 701, 0.17),
    "racism":     (8, 17, 0.47),
}
order = ["homophobia", "insult", "xenophobia", "misogyny", "obscene", "racism"]
print(f"{'categoria':12} {'artigo':>15} {'reproduzido':>15}")
for c in order:
    a = ARTIGO[c]; r = res[c]
    print(f"{c:12} {a[0]:>5}/{a[1]:<3}({a[2]:.2f})  {r['false_negatives']:>5}/{r['positives']:<3}({r['rate']:.2f})")

x = np.arange(len(order)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, [ARTIGO[c][2] for c in order], w, label="Artigo")
ax.bar(x + w/2, [res[c]["rate"] for c in order], w, label="Reproduzido")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20)
ax.set_ylabel("FN-rate"); ax.set_title("Tabela 12 - taxa de falsos negativos por categoria")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES / "tab12_fn_rate.png", dpi=120)
plt.close(fig)
print("figura salva:", FIGURES / "tab12_fn_rate.png")

categoria             artigo     reproduzido
homophobia       7/35 (0.20)      7/35 (0.20)
insult          67/448(0.15)     74/444(0.17)
xenophobia      13/19 (0.68)      9/19 (0.47)
misogyny         7/45 (0.15)      7/44 (0.16)
obscene        117/701(0.17)    119/699(0.17)
racism           8/17 (0.47)      7/17 (0.41)
figura salva: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\figures\tab12_fn_rate.png
